# ALICE RL — Unsloth + TRL GRPO Training (Colab)

Train ALICE benchmark models using **Unsloth 4-bit QLoRA** — 2× faster training
and ~60% less VRAM than standard transformers.

**Features demonstrated:**
- ✅ Unsloth 4-bit QLoRA (NF4, double quantisation)
- ✅ Chain-of-Thought (CoT) prompting with `<reasoning>` tags
- ✅ Three-tier verifier stack (T1 RestrictedPython → T2 LLM judge → T3 regression)
- ✅ Curriculum manager — discrimination zone tasks (20%–80% success rate window)
- ✅ Failure bank — sentence-transformer embeddings, k-NN novelty scoring
- ✅ GRPO group-normalised advantage estimation + KL regularisation
- ✅ Live leaderboard updates (pushes results to `/leaderboard/update`)

**Models:** Qwen2.5-0.5B | Qwen2.5-1.5B | Qwen2.5-3B | SmolLM2-1.7B | Gemma-3-1B

> **Tip:** Enable GPU in Colab: Runtime → Change runtime type → T4 GPU

In [ ]:
# Install Unsloth (auto-detects CUDA version)
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q httpx trl peft accelerate bitsandbytes

In [ ]:
import os

# ── Configuration ────────────────────────────────────────────────────
ALICE_ENV_URL  = "https://rohanjain1648-alice-rl-environment.hf.space"
MODEL_ID       = "Qwen/Qwen2.5-1.5B-Instruct"   # Swap to any benchmark model
EPISODES       = 50
GROUP_SIZE     = 4
MAX_TURNS      = 3
LR             = 5e-6
KL_COEF        = 0.04
MAX_SEQ_LENGTH = 1024
MAX_NEW_TOKENS = 128
LORA_RANK      = 16

os.environ["ALICE_ENV_URL"] = ALICE_ENV_URL
print(f"Environment: {ALICE_ENV_URL}")
print(f"Model:       {MODEL_ID}")

In [ ]:
import httpx, json

try:
    health = httpx.get(f"{ALICE_ENV_URL}/health", timeout=15.0).json()
    print("✅ ALICE Environment healthy:")
    print(json.dumps(health, indent=2))
except Exception as e:
    print(f"❌ Cannot reach environment: {e}")

In [ ]:
import torch

try:
    from unsloth import FastLanguageModel
    print("✅ Unsloth available — loading with 4-bit QLoRA")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_ID,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
        dtype=None,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_RANK,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                         "gate_proj", "up_proj", "down_proj"],
        lora_alpha=LORA_RANK * 2,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
    )
    UNSLOTH_ACTIVE = True
    print("✅ Unsloth 4-bit QLoRA ready")

except ImportError:
    print("⚠️  Unsloth not available — falling back to standard transformers")
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, TaskType
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16,
        ),
        device_map="auto", trust_remote_code=True, torch_dtype=torch.bfloat16,
    )
    model = get_peft_model(model, LoraConfig(
        task_type=TaskType.CAUSAL_LM, r=LORA_RANK,
        lora_alpha=LORA_RANK*2, lora_dropout=0.05,
        target_modules=["q_proj", "v_proj"], bias="none",
    ))
    UNSLOTH_ACTIVE = False
    print("✅ Standard transformers model ready")

model.print_trainable_parameters()

In [ ]:
def env_reset():
    return httpx.post(f"{ALICE_ENV_URL}/reset", timeout=30.0).json()

def env_step(episode_id, action):
    return httpx.post(f"{ALICE_ENV_URL}/step",
                      json={"episode_id": episode_id, "action": action},
                      timeout=30.0).json()

def sample_response(prompt):
    device = next(model.parameters()).device
    # CoT prefix: model reasons before answering
    cot_prompt = f"<task>{prompt}</task>\nReason step by step.\n<reasoning>"
    inputs = tokenizer(cot_prompt, return_tensors="pt", truncation=True, max_length=512).to(device)

    if UNSLOTH_ACTIVE:
        FastLanguageModel.for_inference(model)   # 2× faster Unsloth inference

    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True, temperature=0.8, top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )
    gen_ids = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

def collect_rollouts():
    prompts, responses, rewards = [], [], []
    for _ in range(GROUP_SIZE):
        try:
            ep    = env_reset()
            ep_id = ep["episode_id"]
            task  = ep["task"]
            total_reward = 0.0
            last_action  = ""
            for _ in range(MAX_TURNS):
                action = sample_response(task)
                result = env_step(ep_id, action)
                total_reward += result["reward"]
                last_action   = action
                if result.get("done"): break
            prompts.append(task)
            responses.append(last_action)
            rewards.append(total_reward)
        except Exception as exc:
            print(f"Rollout error: {exc}")
    return prompts, responses, rewards

print("✅ Rollout functions ready")

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
all_rewards, all_successes = [], []

for ep in range(1, EPISODES + 1):
    prompts, responses, rewards = collect_rollouts()
    if not rewards:
        continue

    # Switch to training mode if using Unsloth
    if UNSLOTH_ACTIVE:
        FastLanguageModel.for_training(model)

    # GRPO — group-normalised advantages
    rewards_t  = torch.tensor(rewards, dtype=torch.float32)
    advantages = (rewards_t - rewards_t.mean()) / (rewards_t.std().clamp(min=1e-6))
    device     = next(model.parameters()).device
    total_loss = torch.tensor(0.0, device=device)

    for adv, prompt, response in zip(advantages, prompts, responses):
        inputs     = tokenizer(prompt + "\n" + response, return_tensors="pt",
                               truncation=True, max_length=768).to(device)
        labels     = inputs["input_ids"].clone()
        prompt_len = len(tokenizer(prompt, return_tensors="pt")["input_ids"][0])
        labels[0, :prompt_len] = -100
        out        = model(**inputs, labels=labels)
        log_prob   = -out.loss
        total_loss = total_loss + (-(adv.to(device) * log_prob) + KL_COEF * log_prob**2)

    total_loss = total_loss / max(len(rewards), 1)
    optimizer.zero_grad()
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    all_rewards.extend(rewards)
    all_successes.extend([1.0 if r > 0.3 else 0.0 for r in rewards])
    avg_r    = sum(all_rewards[-50:])   / max(len(all_rewards[-50:]),   1)
    avg_succ = sum(all_successes[-50:]) / max(len(all_successes[-50:]), 1)

    if ep % 5 == 0:
        print(f"Ep {ep:4d} | loss={float(total_loss):.4f} | avg_reward={avg_r:.4f} | success={avg_succ:.2%}")

    if ep % 10 == 0:
        try:
            httpx.post(f"{ALICE_ENV_URL}/leaderboard/update",
                       json={"model_id": MODEL_ID, "avg_reward": avg_r,
                             "success_rate": avg_succ, "discrimination_coverage": 0.0,
                             "episodes_run": ep}, timeout=5.0)
        except Exception: pass

print("\n✅ Training complete!")
print(f"Unsloth active: {UNSLOTH_ACTIVE}")
print(f"Final avg_reward={avg_r:.4f} | success={avg_succ:.2%} | episodes={len(all_rewards)}")

In [ ]:
CKPT_PATH = f"/content/{MODEL_ID.replace('/', '_')}_unsloth_alice"
model.save_pretrained(CKPT_PATH)
tokenizer.save_pretrained(CKPT_PATH)
print(f"Checkpoint saved to {CKPT_PATH}")

# Optional: push to HF Hub
# from huggingface_hub import HfApi
# HfApi().upload_folder(folder_path=CKPT_PATH, repo_id="your-username/alice-trained",
#                       token="hf_...", commit_message="ALICE Unsloth GRPO checkpoint")